# 🚀 LightLLM - Kaggle 30-Hour GPU Training Notebook (MAX VRAM EDITION)
Maximized for 16GB VRAM on Kaggle NVIDIA T4 / P100 GPUs (~14GB+ VRAM Utilization & 32,768 Tokens/Step).

In [ ]:
# Step 1: Verify High-End GPU Hardware & VRAM Capacity
!nvidia-smi

In [ ]:
# Step 2: Clone repository & install dependencies
!git clone https://github.com/RABNEER/LightLLM.git
%cd LightLLM
!pip install torch numpy tiktoken tqdm

In [ ]:
# Step 3: Prepare expanded dataset (Facts, Math, Instructions & Greetings)
!python prepare_data.py

In [ ]:
# Step 4: Configure train.py for MAX VRAM (batch_size=64, ~14GB VRAM usage, 50,000 steps)
config_code = '''
import re
with open('train.py', 'r') as f:
    content = f.read()

# Set batch_size=64 for maximum GPU VRAM throughput (~14GB VRAM)
content = re.sub(r'batch_size\s*=\s*\d+', 'batch_size = 64', content)
# Set max_iters=50000 for long deep training
content = re.sub(r'max_iters\s*=\s*\d+', 'max_iters = 50000', content)
content = re.sub(r'lr_decay_iters\s*=\s*\d+', 'lr_decay_iters = 50000', content)

with open('train.py', 'w') as f:
    f.write(content)
print('[SUCCESS] train.py updated for MAX VRAM (~14GB) & 50,000 steps!')
'''
with open('set_max_vram.py', 'w') as f:
    f.write(config_code)
!python set_max_vram.py

In [ ]:
# Step 5: Launch High-Speed Cloud GPU Training (Max VRAM Throughput)
!python train.py

In [ ]:
# Step 6: Interactive Chat Test
import torch
from lightllm.model import LightLLM
from lightllm.config import LightLLMConfig
from lightllm.tokenizer import Tokenizer

config = LightLLMConfig()
model = LightLLM(config)
tokenizer = Tokenizer()
checkpoint = torch.load('out/checkpoint.pt', map_location='cuda')
model.load_state_dict(checkpoint['model'], strict=False)
model.to('cuda').eval()

def chat(prompt):
    formatted = f"User: {prompt}\nAssistant:"
    ids = torch.tensor([tokenizer.encode(formatted)], dtype=torch.long).to('cuda')
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=60, temperature=0.2, top_k=5)
    return tokenizer.decode(out[0].tolist()).split('<|endoftext|>')[0]

print("Q: hello ->", chat("hello"))
print("Q: 2+2 ->", chat("2+2"))
print("Q: 7*8 ->", chat("7*8"))
print("Q: what is an apple ->", chat("what is an apple"))
print("Q: what is python ->", chat("what is python"))
print("Q: who created you ->", chat("who created you"))

In [ ]:
# Step 7: Save Checkpoint to Kaggle Output Directory
!cp out/checkpoint.pt /kaggle/working/LightLLM_124M_trained.pt
print('[SUCCESS] Trained weights saved to /kaggle/working/LightLLM_124M_trained.pt (Ready to download!)')